# Lesson 18 Lab — TensorRT Sparse Deployment and Polygraphy Evidence

**Puzzle:** What must the build log show before `--sparsity=enable` becomes an acceleration claim?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

TensorRT evaluates structured-sparsity eligibility and tactic profitability. A 2:4-compliant ONNX weight plus a sparsity flag makes a layer eligible; the builder can still select a dense tactic. Polygraphy can help inspect and transform models, but only engine logs and matched benchmarks establish execution.


## 0. Predict before running

1. Predict whether a compliant weight alone proves a sparse tactic ran.
2. List the log lines and numerical checks required for acceptance.
3. Explain why forced pruning must be treated as a new model candidate.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The lab creates a compliant convolution/linear-style weight, checks pattern and dtype gates, probes TensorRT and Polygraphy packages plus `trtexec`, and produces an eligibility-versus-selection matrix.

- Eligibility and tactic selection are separate log events.
- Forcing a pattern is a model mutation that needs quality validation.
- The engine version, flags, profiles, and timing cache identify the build.


## 2. Derive the mechanism

TensorRT's structured sparsity requires the documented local weight pattern and supported FP16 or INT8 execution. The builder reports eligible layers separately from layers for which sparse tactics are chosen. `--sparsity=force` style mutation changes weights and therefore quality; enable mode should consume already compliant weights. Strong typing, shapes, workspace, and version influence tactic search.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 18
LESSON_TITLE = 'TensorRT Sparse Deployment and Polygraphy Evidence'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260826
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | 2:4-compliant BF16/FP16 weight and dense PyTorch numerical control |
| Candidate | native TensorRT build/tactic path when packages and `trtexec` are available |
| Held constant | weight, grouping axis, dtype gate, environment, package probes, and required build fields |
| Measurements | 2:4 compliance, dtype eligibility, TensorRT/Polygraphy/trtexec availability, and native engine status |
| Evidence | `compatibility-probe` |

**Experiment:** Build the full pre-engine eligibility ledger and probe native TensorRT/Polygraphy tools on the RTX 5090 host.


## 5. Read the experiment code

The notebook can prove the data-side invariant on CUDA and can prove whether native tools exist. It cannot infer a TensorRT engine from PyTorch timing. The gate dictionary leaves engine build and sparse-tactic selection false unless those events actually occur.

Do not execute until the code implements the frozen table above.


In [2]:
w=torch.randn(1024,1024,device=DEVICE,dtype=torch.float16); compliant=w*exact_2_4_mask(w)
trt_available=importlib.util.find_spec("tensorrt") is not None; poly_available=importlib.util.find_spec("polygraphy") is not None; trtexec=shutil.which("trtexec")
metrics={"nm_compliance":compliance_2_4(compliant),"sparsity":zero_fraction(compliant),"dtype":str(compliant.dtype),"dtype_eligible":compliant.dtype in (torch.float16,torch.int8),"tensorrt_available":trt_available,"polygraphy_available":poly_available,"trtexec_available":trtexec is not None,"trtexec_path":trtexec,"eligible_weight_data":bool(compliance_2_4(compliant)==1.0),"sparse_engine_built":False,"sparse_tactic_selected":False}
analysis=(f"The weight passed {metrics['nm_compliance']:.1%} of exact 2:4 groups at {metrics['sparsity']:.1%} sparsity and "
          f"used an eligible dtype={metrics['dtype_eligible']}. TensorRT/Polygraphy/trtexec availability was "
          f"{trt_available}/{poly_available}/{trtexec is not None}. Because no engine was built, sparse tactic selection remains false.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| 2:4 compliance | 100.00% |
| Dtype eligible | yes |
| TensorRT available | no |
| Polygraphy available | no |
| trtexec available | no |
| Sparse engine built | no |


## 7. Interpret rather than merely print

The weight passed 100.0% of exact 2:4 groups at 50.0% sparsity and used an eligible dtype=True. TensorRT/Polygraphy/trtexec availability was False/False/False. Because no engine was built, sparse tactic selection remains false.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The notebook records real package/API availability and preserves the native success or failure state. Missing backend execution remains unmeasured.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 18,
    "title": 'TensorRT Sparse Deployment and Polygraphy Evidence',
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'TensorRT sparsity is proven by eligibility, selected tactic, correctness, and benchmark evidence—not by a flag.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 18,
  "title": "TensorRT Sparse Deployment and Polygraphy Evidence",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260826
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "nm_compliance": 1.0,
    "sparsity": 0.5,
    "dtype": "torch.float16",
    "dtype_eligible": true,
    "tensorrt_available": false,
    "polygraphy_available": false,
    "trtexec_available": false,
    "trtexec_path": null,
    "eligible_weight_data": true,
    "sparse_engine_built": false,
    "sparse_tactic_selected": false
  },
  "analysis": "The weight passed 100.0% of exact 2:4 groups at 50.0% sparsity and used an eligible dtype=True. TensorRT/Polygraphy/trtexec availability was False/False/False. Because no engine was built, sparse tactic selection remains false.",
  "conclusion": "TensorRT sparsity is proven by eligibility, selected tactic, cor

## 9. Make the bounded decision

> TensorRT sparsity is proven by eligibility, selected tactic, correctness, and benchmark evidence—not by a flag.

**Acceptance/rollback:** Accept TensorRT sparsity only with a valid engine, explicit eligible and selected sparse-tactic logs, output parity, and matched dense/sparse engine benchmarks.

**Failure analysis:** A builder flag can be ignored by ineligible layers or lose tactic search to a faster dense kernel. Dynamic profiles may select different tactics, and a successful build on A100 does not establish behavior on RTX 5090.


## 10. Extend the evidence

Use a TensorRT-enabled container, export the compliant model, retain Polygraphy inspection plus verbose build logs, and benchmark every production optimization profile.

The full evidence boundary and references are in [`README.md`](README.md).
